In [9]:
import os 

In [10]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    ALL_REQUIRED_FILES: list

In [11]:
from NexText.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from NexText.utils.common import read_yaml, create_directories
from NexText.entity.config_entity import DataValidationConfig

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            ALL_REQUIRED_FILES=config.ALL_REQUIRED_FILES,
        )

        return data_validation_config


In [12]:
import os
from NexText.logging import logger

In [13]:
from pathlib import Path

from NexText.constants import PROJECT_ROOT
from NexText.logging import logger
from NexText.entity.config_entity import DataValidationConfig


class DataValidation:

    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_files_exist(self) -> bool:

        try:
            validation_status = True

            # Dataset is created by Data Ingestion
            data_ingestion_dir = (
                PROJECT_ROOT / "artifacts" / "data_ingestion"
            )

            logger.info(
                f"Checking dataset in: {data_ingestion_dir}"
            )

            dataset_dir = (
                data_ingestion_dir / "samsum_dataset"
                if (data_ingestion_dir / "samsum_dataset").exists()
                else data_ingestion_dir
            )

            for file in self.config.ALL_REQUIRED_FILES:

                file_path = dataset_dir / file

                if not file_path.exists():
                    validation_status = False

                    logger.error(
                        f"Missing required file/folder: {file_path}"
                    )
                else:
                    logger.info(
                        f"Found: {file_path}"
                    )

            # Create validation status file
            status_file = (
                PROJECT_ROOT / self.config.STATUS_FILE
            )

            status_file.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            with open(
                status_file,
                "w",
                encoding="utf-8"
            ) as f:
                f.write(
                    f"Validation status: {validation_status}"
                )

            logger.info(
                f"Data validation status: {validation_status}"
            )

            return validation_status

        except Exception as e:
            logger.exception(e)
            raise e

In [14]:
from NexText.config.configuration import ConfigurationManager
from NexText.components.data_validation import DataValidation

config = ConfigurationManager()

data_validation_config = config.get_data_validation_config()

data_validation = DataValidation(
    config=data_validation_config
)

validation_status = data_validation.validate_all_files_exist()

print("Validation status:", validation_status)

Validation status: False


In [15]:
from NexText.constants import PROJECT_ROOT

data_path = PROJECT_ROOT / "artifacts" / "data_ingestion"

print("Data path:", data_path)
print("Exists:", data_path.exists())

print("\nContents:")
for item in data_path.rglob("*"):
    print(item)

Data path: D:\NLP Project\artifacts\data_ingestion
Exists: True

Contents:
D:\NLP Project\artifacts\data_ingestion\data.zip
D:\NLP Project\artifacts\data_ingestion\samsum-test.csv
D:\NLP Project\artifacts\data_ingestion\samsum-train.csv
D:\NLP Project\artifacts\data_ingestion\samsum-validation.csv
D:\NLP Project\artifacts\data_ingestion\samsum_dataset
D:\NLP Project\artifacts\data_ingestion\samsum_dataset\dataset_dict.json
D:\NLP Project\artifacts\data_ingestion\samsum_dataset\test
D:\NLP Project\artifacts\data_ingestion\samsum_dataset\train
D:\NLP Project\artifacts\data_ingestion\samsum_dataset\validation
D:\NLP Project\artifacts\data_ingestion\samsum_dataset\test\data-00000-of-00001.arrow
D:\NLP Project\artifacts\data_ingestion\samsum_dataset\test\dataset_info.json
D:\NLP Project\artifacts\data_ingestion\samsum_dataset\test\state.json
D:\NLP Project\artifacts\data_ingestion\samsum_dataset\train\data-00000-of-00001.arrow
D:\NLP Project\artifacts\data_ingestion\samsum_dataset\train\dat

In [16]:
print(config.config.data_validation)

{'root_dir': 'artifacts/data_validation', 'STATUS_FILE': 'artifacts/data_validation/status.txt', 'ALL_REQUIRED_FILES': ['train', 'test', 'validation']}
